# Day 24 — A股交易规则与费用

**核心认知**：不了解游戏规则，再好的策略也白搭。

**四条规则**：T+1制度 | 涨跌停板 | 交易费用 | 除权除息

## 一、T+1 制度

- **规则**：今天买的股票，最早明天才能卖
- 周一10:00买入 → 最早周二09:30卖出
- 周五14:00买入 → 最早下周一09:30卖出

**对散户的影响**：盘中追涨风险极大、周五买要谨慎（周末风险）、打板当天炸板没法止损。

## 二、涨跌停板

| 板块 | 限制 | 代码特征 |
|------|------|----------|
| 主板 | ±10% | 60xxxx/00xxxx |
| 科创/创业 | ±20% | 688xxx/300xxx |
| 北交所 | ±30% | 8xxxxx |
| ST股 | ±5% | 含ST |

**对散户的影响**：跌停卖不掉、涨停买不到。T+1+跌停板=双重锁死。

## 三、交易费用

| 费用 | 费率 | 收取 |
|------|------|------|
| 印花税 | 0.05% | 卖出单向 |
| 佣金 | 0.01-0.03% | 买卖双向(最低5元) |
| 过户费 | 0.001% | 买卖双向 |

**关键**：频繁交易=给券商打工。超短线年手续费可达本金的37.5%。

In [1]:
# 交易频率 vs 手续费侵蚀
capital = 100000  # 本金10万
fee_rate = 0.0015  # 来回约0.15%

print(f'本金: {capital:,}元 | 每笔来回手续费约: {fee_rate:.1%}')
print()
print(f'{"策略":20s} {"年交易笔数":>8s} {"年手续费":>10s} {"占本金":>8s}')
print('-' * 50)

scenarios = [
    ('超短线(每天)', 250),
    ('短线(每周2次)', 100),
    ('波段(每两周1次)', 25),
    ('中线(每月1次)', 12),
    ('长线(每季度1次)', 4),
]

for name, trades in scenarios:
    annual_fee = capital * fee_rate * trades
    fee_pct = annual_fee / capital * 100
    print(f'{name:20s} {trades:8d}笔 {annual_fee:8.0f}元 {fee_pct:7.1f}%')

print(f'\n→ 超短线和10天策略一年差{capital*fee_rate*(250-25):.0f}元手续费')

本金: 100,000元 | 每笔来回手续费约: 0.1%

策略                      年交易笔数       年手续费      占本金
--------------------------------------------------
超短线(每天)                   250笔    37500元    37.5%
短线(每周2次)                  100笔    15000元    15.0%
波段(每两周1次)                  25笔     3750元     3.8%
中线(每月1次)                   12笔     1800元     1.8%
长线(每季度1次)                   4笔      600元     0.6%

→ 超短线和10天策略一年差33750元手续费


## 四、除权除息

- **除息**：分现金 → 股价减掉分红金额 → 总资产不变
- **除权**：送股票 → 股价按比例稀释 → 总资产不变

**核心认知**：分红不是天上掉馅饼。是你的钱从左口袋（股票）换到右口袋（现金）。

**三种价格**：
- 不复权：除息日出现假缺口 → ❌ 技术分析不用
- 前复权：历史价格等比调整 → ✅ 技术分析用这个
- 后复权：最新价格等比调整 → 只看累计收益

**红利税**：持股<1月缴20%税 → 短线交易别碰除息日。

In [2]:
# 除权除息：前复权 vs 不复权 对比
print('假设某股票，第4天除息(分红2元)：')
print()
print(f'{"日期":12s} {"不复权":>8s} {"前复权":>8s} {"说明":>25s}')
print('-' * 55)

dates = ['周一','周二','周三','周四(除息)','周五']
raw    = [100, 102, 101, 99, 100]   # 不复权
adj    = [98,  100,  99, 99, 100]   # 前复权(历史减2元)

for d, r, a in zip(dates, raw, adj):
    note = ''
    if '除息' in d:
        note = '← 股价减了2元(分红)'
    elif r != a:
        note = '← 历史价格调低2元'
    print(f'{d:12s} {r:8.0f} {a:8.0f} {note:>25s}')

print()
print('不复权: 周四101→99，看起来跌了，其实是分红导致的')
print('前复权: 历史价格都减2元→ 99→99，平滑过渡，反映真实走势')
print()
print('💡 我们的五只股票数据都是前复权，技术指标不会失真。')

假设某股票，第4天除息(分红2元)：

日期                不复权      前复权                        说明
-------------------------------------------------------
周一                100       98                ← 历史价格调低2元
周二                102      100                ← 历史价格调低2元
周三                101       99                ← 历史价格调低2元
周四(除息)             99       99              ← 股价减了2元(分红)
周五                100      100                          

不复权: 周四101→99，看起来跌了，其实是分红导致的
前复权: 历史价格都减2元→ 99→99，平滑过渡，反映真实走势

💡 我们的五只股票数据都是前复权，技术指标不会失真。


## 五、四条规则 × 我们的策略

| 规则 | 影响 | 应对 |
|------|------|------|
| T+1 | 不能日内纠错 | 收盘前确认信号 |
| 涨跌停 | 极端情况卖不掉 | 单笔风险≤2% |
| 手续费 | 频繁交易费用高 | 持仓10天，低频 |
| 除权除息 | K线假缺口 | 用前复权数据 |

**四条规则刚好支撑10天持仓策略。**

## 六、完整策略（Day20-24汇总）

```
买入信号: MACD金叉（DIF从下穿上DEA）
持仓时间: 10个交易日
止损线:   -5%
仓位:     牛市40% / 震荡30% / 熊市20%
执行:     盘前填计划卡，盘中不碰鼠标
规则:     T+1不追涨、跌停不重仓、低频省手续费、前复权避分红坑
```

**核心认知**：了解游戏规则，才能在场上不犯规。四条规则不是障碍，是保护。